In [ ]:
import pandas as pd
import altair as alt
import numpy as np 

In [ ]:
df = pd.read_csv("../data/congress_generational_summary.csv")

In [ ]:
df1 = pd.read_csv("../data/congress_individual_members.csv")

In [ ]:
print(df1.head())
print("\nColumns:", df1.columns.tolist())

In [ ]:
print("Data loaded successfully:")
print(df.head())

In [ ]:
df.drop(df[df['Generation'] == 'Unknown'].index, inplace=True)  # Drop 'Unknown' generation if it exists

In [ ]:
# Create aggregated data by Generation and Party
df_party = df1[df1['Generation'] != 'Unknown'].groupby(['Generation', 'Party']).size().reset_index(name='Count')

# Map party codes to full names
party_mapping = {'D': 'Democrat', 'R': 'Republican', 'I': 'Independent'}
df_party['Party'] = df_party['Party'].map(party_mapping)

print(df_party)

In [ ]:
generation_viz = alt.Chart(df_party).mark_bar(size=40).encode(
        y=alt.Y('Generation:N', sort=alt.EncodingSortField(field='Count', op='sum', order='descending')).axis(alt.Axis(labelAngle=0)),
        x=alt.X('Count:Q', title='Number of Members'),
        color=alt.Color('Party:N', 
                       scale=alt.Scale(domain=['Democrat', 'Republican', 'Independent'], 
                                     range=['#0015BC', '#FF0000', '#9966CC']),
                       legend=alt.Legend(title='Political Party')),
        tooltip=['Generation', 'Party', 'Count']
    ).configure_axis(
        labelFontSize=12,
        titleFontSize=14,
        grid=False
    ).properties(
        title='Congressional Members by Generation and Party',
        width=400,
        height=300,
    )
generation_viz

## 📸 How to Get Real Representative Photos

To use actual congressional photos instead of placeholders, you have several options:

### **1. Official Congressional Photos (Bioguide)**
```python
# Pattern: https://bioguide.congress.gov/bioguide/photo/[BioID]/[BioID].jpg
# You need the BioguideID for each representative
```

### **2. House.gov Official Photos** 
```python
# Pattern varies by representative's official page
# Example: https://www.house.gov/representatives/find-your-representative
```

### **3. Wikipedia/Wikimedia Commons**
```python
# Many reps have photos on Wikipedia
# Use Wikipedia API to get image URLs
```

### **4. Manual Photo Collection**
You could create a CSV file mapping names to photo URLs:
```csv
Name,PhotoURL
"Smith, John",https://example.com/photos/john_smith.jpg
"Doe, Jane",https://example.com/photos/jane_doe.jpg
```



## 🚀 Using the Congress Photo Fetcher

I've created a separate file `congress_photo_fetcher.py` that extends your `python_analyzer.py` functionality to fetch **real official congressional photos**. Here's how to use it:

In [ ]:
# Import and use the Congress Photo Fetcher
# First, make sure you have the congress_photo_fetcher.py file in the same directory

# Option 1: Run the photo fetcher as a standalone script
# Uncomment the line below to run it in terminal:
# !python congress_photo_fetcher.py

# Option 2: Use it directly in the notebook
try:
    from congress_photo_fetcher import CongressPhotoFetcher
    
    # Initialize the fetcher (make sure CONGRESS_API_KEY is set)
    fetcher = CongressPhotoFetcher()
    
    # Enhance your existing data with official photos
    print("Fetching official congressional photos...")
    df_with_real_photos = fetcher.enhance_congress_data_with_photos("congress_individual_members.csv")
    
    if df_with_real_photos is not None:
        print("Photo fetching complete!")
        
        # Show sample of enhanced data
        print("\nSample of data with official photos:")
        sample_cols = ['Name', 'Party', 'BillCount', 'BioguideID', 'OfficialPhotoURL']
        print(df_with_real_photos[sample_cols].head(3))
        
        # Count official vs fallback photos
        official_count = df_with_real_photos['OfficialPhotoURL'].notna().sum()
        total_count = len(df_with_real_photos)
        print(f"\nStatistics:")
        print(f"   Official photos found: {official_count}/{total_count} ({official_count/total_count*100:.1f}%)")
        print(f"   Fallback photos: {total_count - official_count}")
        
    else:
        print("Could not load photo data. Make sure congress_individual_members.csv exists.")
        
except ImportError:
    print("Error: {e}")

In [ ]:
# Create visualization with REAL congressional photos
# This will use the enhanced dataset with official photos

try:
    # Load the enhanced dataset (created by congress_photo_fetcher.py)
    df_real_photos = pd.read_csv("congress_members_with_photos.csv")
    
    # Create enhanced visualization with official photos
    official_photo_chart = alt.Chart(df_real_photos[df_real_photos['Generation'] != 'Unknown']).mark_circle(
        opacity=0.85,
        stroke='white',
        strokeWidth=1.5
    ).add_params(
        alt.selection_interval(bind='scales')  # Allows zooming and panning
    ).encode(
        x=alt.X('BirthYear:Q', 
               title='Birth Year',
               scale=alt.Scale(domain=[1935, 2000])),
        y=alt.Y('BillCount:Q', 
               title='Bills Sponsored',
               scale=alt.Scale(type='sqrt')),
        color=alt.Color('Party:N',
                       scale=alt.Scale(
                           domain=['Democrat', 'Republican', 'Independent'],
                           range=['#1f77b4', '#d62728', '#9467bd']  # Classic political colors
                       ),
                       legend=alt.Legend(title='Political Party', titleFontSize=12)),
        size=alt.Size('BillCount:Q', 
                     scale=alt.Scale(range=[60, 400], type='sqrt'),
                     legend=alt.Legend(title='Bills Sponsored')),
        tooltip=[
            alt.Tooltip('Name:N', title='Representative'),
            alt.Tooltip('Party:N', title='Party'),
            alt.Tooltip('Generation:N', title='Generation'),
            alt.Tooltip('BirthYear:Q', title='Birth Year'),
            alt.Tooltip('BillCount:Q', title='Bills Sponsored'),
            alt.Tooltip('BioguideID:N', title='Bioguide ID'),
            alt.Tooltip('PhotoURL:N', title='Official Photo')  
        ]
    ).properties(
        title=alt.TitleParams(
            text=['Congressional Members: Legislative Activity & Official Photos',
                  'Hover over representatives to see their official congressional portraits'],
            fontSize=16,
            subtitleFontSize=11,
            anchor='start'
        ),
        width=800,
        height=600
    ).configure_axis(
        labelFontSize=11,
        titleFontSize=12,
        gridOpacity=0.3
    ).configure_legend(
        labelFontSize=11,
        titleFontSize=12
    )
    
    # Display the chart
    print("Features:")
    print("  - Official photos from bioguide.congress.gov")
    print("  - Zoom and pan enabled (drag to zoom, scroll to pan)")
    print("  - Bubble size represents legislative activity")
    print("\nChart saved to: congress_members_with_photos.html")
    
    # Show the chart
    member_activity_enhanced = official_photo_chart
    member_activity_enhanced
    
except FileNotFoundError:
    print("File 'congress_members_with_photos.csv' not found.")
    print("Run congress_photo_fetcher.py first to generate the dataset with photos.")